## KNN Scaling Investigation

In [1]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from src.data import load_data
from src.evaluation import evaluate_model
from src.missingness import introduce_missingness

In [2]:
X_train, X_test, y_train, y_test = load_data()

missing_rates = [0.10, 0.20, 0.30, 0.50]
seeds = [1, 2, 3, 4, 5]

In [3]:
# Unscaled KNN
knn_unscaled = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5))
])

knn_unscaled_lr = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

In [4]:
# Scaled before KNN
knn_scaled_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("imputer", KNNImputer(n_neighbors=5)),
    ("model", LogisticRegression(max_iter=5000))
])

In [5]:
knn_unscaled_rf = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("model", RandomForestClassifier(random_state=42))
])

In [6]:
knn_scaled_rf = Pipeline([
    ("scaler", StandardScaler()),
    ("imputer", KNNImputer(n_neighbors=5)),
    ("model", RandomForestClassifier(random_state=42))
])

In [7]:
X_train_missing = introduce_missingness(
    X_train,
    0.10,
    random_state=1
)

X_test_missing = introduce_missingness(
    X_test,
    0.10,
    random_state=1
)

In [8]:
# Test Unscaled
knn_unscaled_lr = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

knn_unscaled_lr.fit(
    X_train_missing,
    y_train
)

metrics_unscaled = evaluate_model(
    knn_unscaled_lr,
    X_test_missing,
    y_test
)

metrics_unscaled

{'accuracy': 0.9649122807017544,
 'f1': 0.971830985915493,
 'auc': 0.9937169312169312}

In [9]:
# Test Scaled before KNN
knn_scaled_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("imputer", KNNImputer(n_neighbors=5)),
    ("model", LogisticRegression(max_iter=5000))
])

knn_scaled_lr.fit(
    X_train_missing,
    y_train
)

metrics_scaled = evaluate_model(
    knn_scaled_lr,
    X_test_missing,
    y_test
)

metrics_scaled

{'accuracy': 0.956140350877193,
 'f1': 0.965034965034965,
 'auc': 0.9947089947089947}

In [10]:
print("Unscaled:", metrics_unscaled)
print("Scaled:", metrics_scaled)

Unscaled: {'accuracy': 0.9649122807017544, 'f1': 0.971830985915493, 'auc': 0.9937169312169312}
Scaled: {'accuracy': 0.956140350877193, 'f1': 0.965034965034965, 'auc': 0.9947089947089947}


In [11]:
knn_results = []

missing_rates = [0.10, 0.20, 0.30, 0.50]
seeds = [1, 2, 3, 4, 5]

models_knn = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "Random Forest": RandomForestClassifier(random_state=42)
}

for missing_rate in missing_rates:

    for seed in seeds:
        # Create the same missingness pattern
        # for both KNN variants

        X_train_missing = introduce_missingness(
            X_train,
            missing_rate,
            random_state=seed
        )

        X_test_missing = introduce_missingness(
            X_test,
            missing_rate,
            random_state=seed
        )

        for model_name, model in models_knn.items():
            # Unscaled KNN
            if model_name == "Logistic Regression":

                unscaled_pipeline = Pipeline([
                    ("imputer", KNNImputer(n_neighbors=5)),
                    ("scaler", StandardScaler()),
                    ("model", model)
                ])

            else:

                unscaled_pipeline = Pipeline([
                    ("imputer", KNNImputer(n_neighbors=5)),
                    ("model", model)
                ])

            unscaled_pipeline.fit(
                X_train_missing,
                y_train
            )

            metrics = evaluate_model(
                unscaled_pipeline,
                X_test_missing,
                y_test
            )

            knn_results.append({
                "missingness": missing_rate,
                "seed": seed,
                "knn_variant": "Unscaled",
                "model": model_name,
                "accuracy": metrics["accuracy"],
                "f1": metrics["f1"],
                "auc": metrics["auc"]
            })

            # Scaled before KNN
            scaled_pipeline = Pipeline([
                ("scaler", StandardScaler()),
                ("imputer", KNNImputer(n_neighbors=5)),
                ("model", model)
            ])

            scaled_pipeline.fit(
                X_train_missing,
                y_train
            )

            metrics = evaluate_model(
                scaled_pipeline,
                X_test_missing,
                y_test
            )

            knn_results.append({
                "missingness": missing_rate,
                "seed": seed,
                "knn_variant": "Scaled",
                "model": model_name,
                "accuracy": metrics["accuracy"],
                "f1": metrics["f1"],
                "auc": metrics["auc"]
            })


knn_results_df = pd.DataFrame(knn_results)

knn_results_df

,missingness,seed,knn_variant,model,accuracy,f1,auc
0,0.1,1,Unscaled,Logistic Regression,0.964912,0.971831,0.993717
1,0.1,1,Scaled,Logistic Regression,0.956140,0.965035,0.994709
2,0.1,1,Unscaled,Random Forest,0.938596,0.951049,0.992890
3,0.1,1,Scaled,Random Forest,0.947368,0.958333,0.992725
4,0.1,2,Unscaled,Logistic Regression,0.964912,0.971831,0.996693
...,...,...,...,...,...,...,...
75,0.5,4,Scaled,Random Forest,0.912281,0.928571,0.984623
76,0.5,5,Unscaled,Logistic Regression,0.894737,0.916667,0.969907
77,0.5,5,Scaled,Logistic Regression,0.938596,0.951049,0.993717
78,0.5,5,Unscaled,Random Forest,0.894737,0.918919,0.971561


In [12]:
knn_summary = (
    knn_results_df
    .groupby(
        ["missingness", "knn_variant", "model"]
    )
    .agg(
        n_runs=("accuracy", "count"),
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        mean_f1=("f1", "mean"),
        std_f1=("f1", "std"),
        mean_auc=("auc", "mean"),
        std_auc=("auc", "std")
    )
    .reset_index()
)

knn_summary

,missingness,knn_variant,model,n_runs,mean_accuracy,std_accuracy,mean_f1,std_f1,mean_auc,std_auc
0,0.1,Scaled,Logistic Regression,5,0.971930,0.011437,0.977758,0.009144,0.994312,0.001468
1,0.1,Scaled,Random Forest,5,0.942105,0.010002,0.953922,0.008302,0.992526,0.001055
2,0.1,Unscaled,Logistic Regression,5,0.968421,0.004805,0.974785,0.003870,0.994643,0.001468
3,0.1,Unscaled,Random Forest,5,0.942105,0.010002,0.953922,0.008302,0.992460,0.000568
4,0.2,Scaled,Logistic Regression,5,0.968421,0.013303,0.974999,0.010534,0.995172,0.002547
5,0.2,Scaled,Random Forest,5,0.938596,0.012405,0.951259,0.010220,0.991369,0.001453
6,0.2,Unscaled,Logistic Regression,5,0.952632,0.020194,0.962018,0.016409,0.990873,0.004061
7,0.2,Unscaled,Random Forest,5,0.933333,0.007846,0.947044,0.006537,0.989087,0.002497
8,0.3,Scaled,Logistic Regression,5,0.959649,0.022874,0.967586,0.019090,0.994577,0.002672
9,0.3,Scaled,Random Forest,5,0.945614,0.013011,0.956738,0.010481,0.991204,0.004453


In [13]:
knn_results_df.groupby(
    ["missingness", "knn_variant", "model"]
).size()

missingness  knn_variant  model              
0.1          Scaled       Logistic Regression    5
                          Random Forest          5
             Unscaled     Logistic Regression    5
                          Random Forest          5
0.2          Scaled       Logistic Regression    5
                          Random Forest          5
             Unscaled     Logistic Regression    5
                          Random Forest          5
0.3          Scaled       Logistic Regression    5
                          Random Forest          5
             Unscaled     Logistic Regression    5
                          Random Forest          5
0.5          Scaled       Logistic Regression    5
                          Random Forest          5
             Unscaled     Logistic Regression    5
                          Random Forest          5
dtype: int64

In [17]:
knn_results_df.to_csv("results/knn_scaling_results.csv", index=False)

## Conclusion
The results show that feature scaling before KNN imputation can have a noticeable effect on downstream model performance. At 10% missingness, the difference between scaled and unscaled KNN was relatively small, but the performance gap increased as the level of missingness increased. This was particularly evident for Logistic Regression, where scaling before KNN improved mean accuracy from 90.7% to 94.6% at 50% missingness. A similar improvement was observed for Random Forest at higher missingness levels.

These results suggest that feature scaling should be considered before KNN imputation because the scaling of features affects the distance calculations used to identify neighbouring observations. Therefore, for the main experiment, KNN imputation will be performed after standardising the features, with the scaler fitted only on the training data to avoid data leakage.